# SPY 0DTE Credit Spreads — ORB-direction

**Setup:** Opening range = first 30 min. First 1-min close outside OR between 10:00 ET and 13:00 ET (10:00 AM PT) opens a credit spread in the *continuation* direction:

- **Break above ORH → bull put spread**: short put 1 OTM, long put 1 further OTM. Profits if SPY stays above the short put.
- **Break below ORL → bear call spread**: mirror. Profits if SPY stays below the short call.

**Capital deployed** = `(spread_width − entry_credit) × 100 × qty` = max loss. PT/SL are measured as a % of this.

**Sweep:** 7 PT × 3 SL × 3 time stops = **63 configs** over the cached year, ~5 min on warm cache.

## 1. Setup

In [ ]:
import os, sys
REPO = '/content/iron-condor'
BRANCH = 'claude/spy-options-trading-bot-ri4Ms'
if not os.path.isdir(REPO):
    !git clone --quiet -b {BRANCH} https://github.com/coolin815/iron-condor.git {REPO}
else:
    !cd {REPO} && git pull --quiet
SRC = REPO + '/src'
os.environ['PYTHONPATH'] = SRC + os.pathsep + os.environ.get('PYTHONPATH', '')
if SRC not in sys.path:
    sys.path.insert(0, SRC)
os.chdir(REPO)
print('OK')

## 2. Paste your Polygon key

In [ ]:
import os, getpass
os.environ['POLYGON_API_KEY'] = getpass.getpass('Polygon API key: ')
print('Key loaded:', len(os.environ['POLYGON_API_KEY']), 'chars')

## 3. Full sweep (63 configs over a year)
First run will fetch new option contracts (short + long strikes for each day's signal) — expect 10–20 min. Subsequent runs are fast.

In [ ]:
!python -m iron_condor.cli --days 365 --sweep

## 4. Top configs

In [ ]:
import pandas as pd
summary = pd.read_csv('results/sweep_summary.csv')
summary.head(20)

## 5. PT × SL heatmaps
Which combo of profit target and stop loss has the best expectancy?

In [ ]:
import re
rows = []
for _, r in summary.iterrows():
    m = re.match(r'pt(\d+)\|sl(\d+)\|ts(\d+)', r['config'])
    if m:
        rows.append({
            'pt': int(m.group(1)),
            'sl': int(m.group(2)),
            'ts': int(m.group(3)),
            'return': r['total_return_pct'],
            'win_rate': r['win_rate'],
            'trades': r['n_trades'],
        })
grid = pd.DataFrame(rows)
print('=== Return % by PT × SL (averaged over time stops) ===')
print(grid.pivot_table(index='pt', columns='sl', values='return', aggfunc='mean').round(1))
print('\n=== Win rate by PT × SL (averaged) ===')
print(grid.pivot_table(index='pt', columns='sl', values='win_rate', aggfunc='mean').round(2))

## 6. Direction & exit-reason breakdown for the top config

In [ ]:
trades = pd.read_csv('results/sweep_trades.csv')
top_cfg = summary.iloc[0]['config']
sub = trades[trades['config'] == top_cfg]
taken = sub[sub['exit_reason'].isin(['profit', 'stop', 'time_stop'])]
print(f'Top config: {top_cfg}')
print('\n=== By direction (bull_put = SPY broke up; bear_call = broke down) ===')
print(taken.groupby('signal_direction').agg(
    trades=('net_pnl', 'count'),
    win_rate=('net_pnl', lambda s: (s > 0).mean()),
    avg_pnl=('net_pnl', 'mean'),
    total_pnl=('net_pnl', 'sum'),
).round(2))
print('\n=== Exit reasons ===')
print(sub['exit_reason'].value_counts())

## 7. Equity curve — top 3 configs

In [ ]:
import matplotlib.pyplot as plt
top3 = summary.head(3)['config'].tolist()
fig, ax = plt.subplots(figsize=(10, 5))
for cfg in top3:
    sub = trades[trades['config'] == cfg].sort_values('day')
    ax.plot(pd.to_datetime(sub['day']), sub['balance_after'], label=cfg, linewidth=1.5)
ax.axhline(1500, color='gray', linestyle='--', alpha=0.5)
ax.set_xlabel('Date'); ax.set_ylabel('Balance ($)')
ax.set_title('Top-3 credit-spread config equity curves')
ax.legend(fontsize=8); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()